# HALO-RAG pipeline walkthrough

This notebook loads the project configuration, prepares a small dataset sample, and runs one verified generation. Model and dataset assets are downloaded on first use.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import yaml

from src.data import load_dataset_from_config, prepare_for_experiments
from src.pipeline import SelfVerificationRAGPipeline
from src.utils.device import resolve_device

with (project_root / 'config/config.yaml').open() as stream:
    config = yaml.safe_load(stream)

device = resolve_device(config['experiments'].get('device', 'auto'))
device

In [ ]:
examples = load_dataset_from_config(config, split='validation')[:10]
queries, ground_truths, relevant_docs, corpus = prepare_for_experiments(examples)
len(queries), len(corpus)

In [ ]:
pipeline = SelfVerificationRAGPipeline(
    corpus=corpus,
    device=device,
    use_qlora=config['generation']['qlora']['training_enabled'],
    enable_revision=True,
)
result = pipeline.generate(queries[0])
result